Imports and Configuration

In [ ]:
import pandas as pd
import pyodbc
from sqlalchemy import create_engine,inspect
from urllib.parse import quote_plus

# --- CONFIGURATION ---
PG_DB = {
    "user": "<username_name>",
    "pass": "<password>",
    "host": "<host_name>",
    "port": "5432",
    "db": "<database_name>"
}

SS_DB = {
    "server": "<server_name>",
    "user": "<user_name>",  # Your SQL Server username (e.g., 'sa')
    "pass": "<password>",
    "db": "<database_name>",
    "driver": "ODBC Driver 18 for SQL Server" 
}

print("Libraries imported and config set")

## Create Engines and Test Connections
- verifies that both databases are reachable before you start moving data

In [ ]:
# URL-encode the password to handle the '@' and '!' characters
pg_pass_encoded = quote_plus(PG_DB['pass'])

# Build the URL using the encoded password
pg_url = f"postgresql://{PG_DB['user']}:{pg_pass_encoded}@{PG_DB['host']}:{PG_DB['port']}/{PG_DB['db']}"

# Create the engine
pg_engine = create_engine(pg_url)

# sql server connection string
ss_conn_str = (
    f"DRIVER={{{SS_DB['driver']}}};"
    f"SERVER={SS_DB['server']};"
    f"DATABASE={SS_DB['db']};"
    f"UID={SS_DB['user']};"
    f"PWD={SS_DB['pass']};"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

ss_engine = create_engine(f"mssql+pyodbc:///?odbc_connect={ss_conn_str}")

# Test Connections
try:
    with pg_engine.connect() as conn:
        print("PostgreSQL Connection Successful")
    with ss_engine.connect() as conn:
        print("SQL Server Connection Successful")
except Exception as e:
    print(f"Connection Error: {e}")

## Inspect Tables
- discover all tables in Postgres schema

In [ ]:
inspector = inspect(pg_engine)
all_tables = inspector.get_table_names(schema='staging')

print(f"Tables found for migration: {all_tables}")

## The Migration Loop
- iterates through the tables and provides a visual progress update

In [ ]:
for table in all_tables:
    try:
        print(f"Reading {table}...")
        # Read from postgresql database
        df = pd.read_sql_table(table, pg_engine, schema='staging')

        if df.empty:
            print(f"Table {table} is empty. Skipping...")
            continue

        # append the prefix for the destination
        destination_name = f"stg_{table}"
        
        # Write to SS (if_exists='replace' creates the DDL/Table automatically)
        print(f"Writing {table} to SQL Server ({len(df)} rows)...")
        df.to_sql(
            destination_name, 
            ss_engine, 
            if_exists='replace', 
            index=False, 
            chunksize=1000
        )

        print(f"Done with {table}")
        
    except Exception as e:
        print(f"Failed to migrate {table}: {e}")

print("\n--- ALL TASKS COMPLETE ---")